import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from lifelines import CoxPHFitter
import pandas as pd
metadata= pd.read_csv('data/combined_metadata.csv')
y= metadata[['geo_accession','metastasis_status','time']].set_index('geo_accession')
X = pd.read_csv('data/combined_expression_data_scaled.csv', index_col=0).convert_dtypes().transpose()
def fit_and_score_features2(X):
   y=X[["event_observed","duration"]]
   X.drop(["duration", "event_observed"], axis=1, inplace=True)
   n_features = X.shape[1]
   scores = np.empty(n_features)
   m = CoxPHFitter()

   for j in range(n_features):
       Xj = X.iloc[:, j:j+1]
       Xj=pd.merge(Xj, y,  how='right', left_index=True, right_index=True)
       m.fit(Xj, duration_col="duration", event_col="event_observed", show_progress=True)
       scores[j] = m.score_
   return scores

def fit_and_score_features(X, y):
    n_features = X.shape[1]
    scores = []
    # Fit a CoxPH model for each feature
    for j in range(n_features):
        Xj = X.iloc[:, j : j + 1] # Select one feature

        try:
            Xj = pd.merge(Xj, y, left_index=True, right_index=True)
            #Xj= Xj['1007_s_at'].drop_duplicates()
            m = CoxPHFitter()
            m.fit(Xj, 'time','metastasis_status' )
            scores.append(m.summary)
        except Exception as e:
            scores[j] = 0.0
    return scores

scores = fit_and_score_features(X, y)
df=pd.concat(scores, ignore_index=True)
df.index = X.columns
df.head()
df.to_csv('lifeline_univ_cox_pvals.csv')
plt.hist(df.p, bins=40)
sig = df[df.p<=0.05].shape[0]
print("precent sig", sig / len(df))
sig
sig_ids = df[df.p<=0.05].index.to_list()
sig_expression = X.loc[:,sig_ids]
sig_expression.to_csv('./data/lifeline_univ_cox_sig_expression.csv')
# MIC
expression_data = pd.read_csv('data/combined_expression_data_scaled.csv')
metadata = pd.read_csv('data/combined_metadata.csv')
mic_data = pd.read_csv('data/mic/mic_c.csv', header=None)
gene_feature_scores = pd.read_csv('data/feature_scores_with_annotations.csv')
#add column names to mic_data
mic_data.columns = ['mic_metastasis_status', 'mic_time']
mic_data['gene'] = gene_feature_scores['Gene Symbol']
mic_data['sum_mic'] = mic_data['mic_metastasis_status'] + mic_data['mic_time']
mic_data['id'] = gene_feature_scores['id']
mic_data.head()
genes_of_interest = ['CDH3', 'ADORA2B', 'MMP14', 'IP6K2', 'HTR2A']

#check mic values for genes of interest
mic_data[mic_data['gene'].isin(genes_of_interest)]
## should probobly only look at metastasis
# make histogram of mic values
plt.hist(mic_data['mic_metastasis_status'], bins=50, alpha=0.5, label='Metastasis Status MIC')
plt.hist(mic_data['mic_time'], bins=50, alpha=0.5, label='Time MIC')
plt.hist(mic_data['sum_mic'], bins=50, alpha=0.5, label='Sum MIC')
plt.legend()
plt.show()
# get the top 20% of genes by mic
top_20_percent_threshold = mic_data['sum_mic'].quantile(0.8)
top_genes = mic_data[mic_data['sum_mic'] >= top_20_percent_threshold]['gene']
top_ids = mic_data[mic_data['sum_mic'] >= top_20_percent_threshold]['id']
print("choosing ", len(top_genes), "out of ", len(mic_data), " genes as top 20% by MIC")

# filter expression data to only include top genes by mic
expression_data_top_genes = expression_data[expression_data.index.isin(top_ids)]
expression_data_top_genes.shape
# save expression data with only top genes by mic
expression_data_top_genes.to_csv('./data/combined_expression_data_top20_mic.csv', index=True)
# retry fitting with top genes
#load data
expression_data_top_genes = pd.read_csv('./data/combined_expression_data_top20_mic.csv', index_col=0)
metadata = pd.read_csv('data/combined_metadata.csv')
# reimport survival functions
from importlib import reload

import survival_functions
reload(survival_functions)
from survival_functions import *
y = Surv.from_arrays(event=metadata['metastasis_status'], time=metadata['time'])
X = expression_data_top_genes.T

pipeline = SurvivalModelPipeline(random_state=42)
pipeline.load_and_prepare_data(X=X, y=y)

pipeline.train_and_evaluate()
pipeline.plot_results()
    
# Print summary table
print("\n" + "="*60)
print("Summary Results")
print("="*60)
summary = pipeline.get_summary()
print(summary.to_string(index=False))

# Get best model
best_model_name = summary.iloc[0]['Model']
print(f"\nBest model by C-index: {best_model_name}")
rsf = pipeline.models['RSF']
#extract feature importances using permutation importance
from sklearn.inspection import permutation_importance
result = permutation_importance(rsf, 
                                pipeline.X_test, 
                                pipeline.y_test, n_repeats=10, random_state=42)
result_df = pd.DataFrame(
    {
        k: result[k]
        for k in (
            "importances_mean",
            "importances_std",
        )
    },
    index=pipeline.X_test.columns,
).sort_values(by="importances_mean", ascending=False)

result_df.sort_values(by="importances_mean", ascending=False).head(20)
lasso =pipeline.models['CoxLasso']

lasso.

In [ ]:
# RSF for top 20% genes
y = Surv.from_arrays(event=metadata['metastasis_status'], time=metadata['time'])
X = expression_data_top_genes.T

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, train_size=0.2, random_state=42
)
# Initialize and fit Random Survival Forest
rsf = RandomSurvivalForest(
    n_estimators=100,           # Number of trees
    min_samples_split=10,       # Minimum samples to split a node
    min_samples_leaf=5,         # Minimum samples in a leaf
    max_features='sqrt',        # Number of features to consider for splitting
    n_jobs=-1,                  # Use all available cores
    random_state=42
)
#skip scalling as we already did?

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Fitting Random Survival Forest...")
rsf.fit(X_train_scaled, y_train)

# Evaluate model performance (C-index)
train_score = rsf.score(X_train_scaled, y_train)
test_score = rsf.score(X_test_scaled, y_test)
print(f"\nModel Performance (C-index):")
print(f"Training score: {train_score:.4f}")
print(f"Testing score: {test_score:.4f}")


# get feature importances
# rsf.feature_importances_ is not implemented for RandomSurvivalForest
# they suggest using permutation importance instead
from sklearn.inspection import permutation_importance

result = permutation_importance(rsf, X_train_scaled, y_train, n_repeats=20, random_state=42, n_jobs=5, max_samples=0.5)
importance = pd.DataFrame(
    {
        k: result[k]
        for k in (
            "importances_mean",
            "importances_std",
        )
    },
    index=X_test.columns,
).sort_values(by="importances_mean", ascending=False)
# load gene annotations
gene_info = pd.read_csv('./data/gene_feature_scores_annotation.csv')
gene_info.head()
importance.head()
# and join with importance scores
importance = importance.reset_index().rename(columns={'ID_REF':'id'})
importance_with_annotations = pd.merge(importance, gene_info, on='id', how='left')
importance_with_annotations.head()
# get top 40 genes
top_40_genes = importance_with_annotations.head(40)
top_40_genes[['Gene Symbol', 'importances_mean']]
top_genes_paper = [
    'S1PR1',
    'MIF',
    'H1R1B',
    'HDS17B10',
    'ABCB4',
    'MAOA']
genes_of_interest = ['CDH3', 'ADORA2B', 'MMP14', 'IP6K2', 'HTR2A']
# Check the ranking of these genes in the importance_with_annotations dataframe
importance_with_annotations[importance_with_annotations['Gene Symbol'].isin(top_genes_paper + genes_of_interest)][['Gene Symbol', 'importances_mean']]

plot =plt.hist(importance_with_annotations.importances_mean, bins=40)
plt.yscale('log')
print(gene_info['Gene Symbol'].duplicated().sum())
print(gene_info.shape)
print(gene_info['Gene Symbol'].duplicated().sum()/gene_info.shape)
#lets cut down the tags by taking the mean of everything with same gene name
#expression_data_scaled = pd.read_csv('data/combined_expression_data_scaled.csv', index_col=0)
expression_data = pd.read_csv('data/combined_expression_data_unscaled.csv', index_col=0)
expression_data_scaled.head()
gene_info.head()
expression_merge = pd.merge(expression_data_scaled,
    gene_info[['ID','Gene Symbol']],
    left_index=True, right_on='ID', how='left'
    )

# now group by gene and take average
expression_merge_avg = expression_merge.drop(columns='ID').groupby('Gene Symbol').mean()
print(expression_merge_avg.shape)
expression_merge_avg.head()
#save
expression_merge_avg.to_csv('data/scaled_expession_by_gene.csv')
c_index = pd.read_csv('data/feature_scores_with_annotations.csv', index_col=0)
plt.hist(c_index['c-index_score'], bins=30)
# filtering by independent c index sxores
# making low cutoff smaller to prevent removing all mitigating genes
threshold_low = 0.05
threshold_high = 0.15
cut_out = (c_index['c-index_score'] > 0.5 - threshold_low) & (c_index['c-index_score'] < 0.5 +threshold_high)
plt.hist(c_index[~cut_out]['c-index_score'], bins=30)
print("original size", c_index.shape)
print("after", c_index[~cut_out].shape)
print(f"removed {cut_out.sum()/len(c_index)}")

# reload expression data and remove these
gene_expression_data = pd.read_csv('data/scaled_expession_by_gene.csv', index_col=0)

genes_keep = c_index[~cut_out]['Gene Symbol'].to_list()
genes_c_index_filtered= gene_expression_data[gene_expression_data.index.isin(genes_keep)]
genes_c_index_filtered.shape
genes_c_index_filtered.to_csv('data/gene_expression_filtered_c_index.csv')
gene_expression_data.shape

# Pairwise MIC clustering
I think that actualy they did feature selection on the pairwise MIC, clustering the results, and using representative genes from each cluster.
this is outlined in 
[Sci-Kit: Permutation Importance with Multicollinear or Correlated Features](https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance_multicollinear.html#sphx-glr-auto-examples-inspection-plot-permutation-importance-multicollinear-py)
the pairwise result can be found by formula
$k = m*i - i*(i+1)/2 - i - 1 + j$

In the example they use Pearsons Correlation to calculate distance, in the paper they use the MIC. So I will be using $D(i,j)= 1-MIC_{i,j}$


from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
metadata= pd.read_csv('data/combined_metadata.csv')
y = Surv.from_arrays(event=metadata['metastasis_status'], time=metadata['time'])
X = pd.read_csv('data/lifeline_univ_cox_sig_expression.csv', index_col=0).convert_dtypes()

# load in MIC pairwise data
mic_pairwise = pd.read_csv('./data/mic/pairwise_results.csv',index_col=0 ).convert_dtypes()
#mic_pairwise = pd.to_numeric(mic_pairwise)
distance_matrix = 1- mic_pairwise
#fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8))

#not technicly correlation but similar idea
#corr = 1 - distance_matrix
#dist_linkage = hierarchy.ward(squareform(distance_matrix))

from sklearn_extra.cluster import KMedoids
n_clusters=500
kmedoids = KMedoids(n_clusters=n_clusters, random_state=42,metric='precomputed')
kmedoids.fit(distance_matrix)
# Get cluster assignments
labels = kmedoids.labels_

# Get the medoids
medoids = kmedoids.cluster_centers_

indices = kmedoids.medoid_indices_
print(indices)
print(labels.shape)
X.iloc[:,indices].shape

X_sel = X.iloc[:,indices]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y, test_size=0.2, random_state=42
)




# Initialize and fit Random Survival Forest
rsf = RandomSurvivalForest(
    n_estimators=100,           # Number of trees
    min_samples_split=10,       # Minimum samples to split a node
    min_samples_leaf=5,         # Minimum samples in a leaf
    max_features='sqrt',        # Number of features to consider for splitting
    n_jobs=-1,                  # Use all available cores
    random_state=42
)
print("Fitting Random Survival Forest...")
rsf.fit(X_train, y_train)

# Evaluate model performance (C-index)
train_score = rsf.score(X_train, y_train)
test_score = rsf.score(X_test, y_test)
print(f"\nModel Performance (C-index):")
print(f"Training score: {train_score:.4f}")
print(f"Testing score: {test_score:.4f}")


#extract feature importances using permutation importance
from sklearn.inspection import permutation_importance
result = permutation_importance(rsf, 
                                X_test, 
                                y_test, n_repeats=16, random_state=42, n_jobs=8)

result_df = pd.DataFrame(
    {
        k: result[k]
        for k in (
            "importances_mean",
            "importances_std",
        )
    },
    index=X_test.columns,
)

result_df.sort_values(by="importances_mean", ascending=False).head(20)
plt.hist(result_df.importances_mean)
## add the importance of cluster to each gene
importance_df = pd.DataFrame([X.columns,labels]).T
importance_df = importance_df.rename(columns={0:'Gene Symbol', 1:'cluster_id'})
importance_df.head()
# add scores from importance
score = result_df.importances_mean.to_list()
cluster_id = range(n_clusters)
score_map = dict(zip(cluster_id,score))

importance_df['feature_importance'] = importance_df['cluster_id'].map(score_map)
importance_df.head()
top_genes_paper = [
    'S1PR1',
    'MIF',
    'H1R1B',
    'HDS17B10',
    'ABCB4',
    'MAOA',
    'OPRK1',
    'IP6K2',
    'TRPA1',
    'CTP11B1',
    'ABCC8',
    'DPAGT1',
    'KDR']
genes_of_interest = ['CDH3', 'ADORA2B', 'MMP14', 'IP6K2', 'HTR2A']
# Check the ranking of these genes in the importance_with_annotations dataframe
importance_df[importance_df['Gene Symbol'].isin(top_genes_paper)][['Gene Symbol', 'feature_importance']]

importance_df.sort_values(by='feature_importance', ascending=False).to_csv('feature_permute_scores.csv')

# Next steps
* run random forest on selected genes

In [ ]:
# RSF for top 20% genes
y = Surv.from_arrays(event=metadata['metastasis_status'], time=metadata['time'])
X = expression_data_top_genes.T

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, train_size=0.2, random_state=42
)
# Initialize and fit Random Survival Forest
rsf = RandomSurvivalForest(
    n_estimators=100,           # Number of trees
    min_samples_split=10,       # Minimum samples to split a node
    min_samples_leaf=5,         # Minimum samples in a leaf
    max_features='sqrt',        # Number of features to consider for splitting
    n_jobs=-1,                  # Use all available cores
    random_state=42
)
#skip scalling as we already did?

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Fitting Random Survival Forest...")
rsf.fit(X_train_scaled, y_train)

# Evaluate model performance (C-index)
train_score = rsf.score(X_train_scaled, y_train)
test_score = rsf.score(X_test_scaled, y_test)
print(f"\nModel Performance (C-index):")
print(f"Training score: {train_score:.4f}")
print(f"Testing score: {test_score:.4f}")


# get feature importances
# rsf.feature_importances_ is not implemented for RandomSurvivalForest
# they suggest using permutation importance instead
from sklearn.inspection import permutation_importance

result = permutation_importance(rsf, X_train_scaled, y_train, n_repeats=20, random_state=42, n_jobs=5, max_samples=0.5)
importance = pd.DataFrame(
    {
        k: result[k]
        for k in (
            "importances_mean",
            "importances_std",
        )
    },
    index=X_test.columns,
).sort_values(by="importances_mean", ascending=False)
# load gene annotations
gene_info = pd.read_csv('./data/gene_feature_scores_annotation.csv')
gene_info.head()
importance.head()
# and join with importance scores
importance = importance.reset_index().rename(columns={'ID_REF':'id'})
importance_with_annotations = pd.merge(importance, gene_info, on='id', how='left')
importance_with_annotations.head()
# get top 40 genes
top_40_genes = importance_with_annotations.head(40)
top_40_genes[['Gene Symbol', 'importances_mean']]
top_genes_paper = [
    'S1PR1',
    'MIF',
    'H1R1B',
    'HDS17B10',
    'ABCB4',
    'MAOA']
genes_of_interest = ['CDH3', 'ADORA2B', 'MMP14', 'IP6K2', 'HTR2A']
# Check the ranking of these genes in the importance_with_annotations dataframe
importance_with_annotations[importance_with_annotations['Gene Symbol'].isin(top_genes_paper + genes_of_interest)][['Gene Symbol', 'importances_mean']]

plot =plt.hist(importance_with_annotations.importances_mean, bins=40)
plt.yscale('log')
print(gene_info['Gene Symbol'].duplicated().sum())
print(gene_info.shape)
print(gene_info['Gene Symbol'].duplicated().sum()/gene_info.shape)
#lets cut down the tags by taking the mean of everything with same gene name
#expression_data_scaled = pd.read_csv('data/combined_expression_data_scaled.csv', index_col=0)
expression_data = pd.read_csv('data/combined_expression_data_unscaled.csv', index_col=0)
expression_data_scaled.head()
gene_info.head()
expression_merge = pd.merge(expression_data_scaled,
    gene_info[['ID','Gene Symbol']],
    left_index=True, right_on='ID', how='left'
    )

# now group by gene and take average
expression_merge_avg = expression_merge.drop(columns='ID').groupby('Gene Symbol').mean()
print(expression_merge_avg.shape)
expression_merge_avg.head()
#save
expression_merge_avg.to_csv('data/scaled_expession_by_gene.csv')
c_index = pd.read_csv('data/feature_scores_with_annotations.csv', index_col=0)
plt.hist(c_index['c-index_score'], bins=30)
# filtering by independent c index sxores
# making low cutoff smaller to prevent removing all mitigating genes
threshold_low = 0.05
threshold_high = 0.15
cut_out = (c_index['c-index_score'] > 0.5 - threshold_low) & (c_index['c-index_score'] < 0.5 +threshold_high)
plt.hist(c_index[~cut_out]['c-index_score'], bins=30)
print("original size", c_index.shape)
print("after", c_index[~cut_out].shape)
print(f"removed {cut_out.sum()/len(c_index)}")

# reload expression data and remove these
gene_expression_data = pd.read_csv('data/scaled_expession_by_gene.csv', index_col=0)

genes_keep = c_index[~cut_out]['Gene Symbol'].to_list()
genes_c_index_filtered= gene_expression_data[gene_expression_data.index.isin(genes_keep)]
genes_c_index_filtered.shape
genes_c_index_filtered.to_csv('data/gene_expression_filtered_c_index.csv')
gene_expression_data.shape

# Pairwise MIC clustering
I think that actualy they did feature selection on the pairwise MIC, clustering the results, and using representative genes from each cluster.
this is outlined in 
[Sci-Kit: Permutation Importance with Multicollinear or Correlated Features](https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance_multicollinear.html#sphx-glr-auto-examples-inspection-plot-permutation-importance-multicollinear-py)
the pairwise result can be found by formula
$k = m*i - i*(i+1)/2 - i - 1 + j$

In the example they use Pearsons Correlation to calculate distance, in the paper they use the MIC. So I will be using $D(i,j)= 1-MIC_{i,j}$


from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
metadata= pd.read_csv('data/combined_metadata.csv')
y = Surv.from_arrays(event=metadata['metastasis_status'], time=metadata['time'])
X = pd.read_csv('data/lifeline_univ_cox_sig_expression.csv', index_col=0).convert_dtypes()

# load in MIC pairwise data
mic_pairwise = pd.read_csv('./data/mic/pairwise_results.csv',index_col=0 ).convert_dtypes()
#mic_pairwise = pd.to_numeric(mic_pairwise)
distance_matrix = 1- mic_pairwise
#fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8))

#not technicly correlation but similar idea
#corr = 1 - distance_matrix
#dist_linkage = hierarchy.ward(squareform(distance_matrix))

from sklearn_extra.cluster import KMedoids
n_clusters=500
kmedoids = KMedoids(n_clusters=n_clusters, random_state=42,metric='precomputed')
kmedoids.fit(distance_matrix)
# Get cluster assignments
labels = kmedoids.labels_

# Get the medoids
medoids = kmedoids.cluster_centers_

indices = kmedoids.medoid_indices_
print(indices)
print(labels.shape)
X.iloc[:,indices].shape

X_sel = X.iloc[:,indices]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y, test_size=0.2, random_state=42
)




# Initialize and fit Random Survival Forest
rsf = RandomSurvivalForest(
    n_estimators=100,           # Number of trees
    min_samples_split=10,       # Minimum samples to split a node
    min_samples_leaf=5,         # Minimum samples in a leaf
    max_features='sqrt',        # Number of features to consider for splitting
    n_jobs=-1,                  # Use all available cores
    random_state=42
)
print("Fitting Random Survival Forest...")
rsf.fit(X_train, y_train)

# Evaluate model performance (C-index)
train_score = rsf.score(X_train, y_train)
test_score = rsf.score(X_test, y_test)
print(f"\nModel Performance (C-index):")
print(f"Training score: {train_score:.4f}")
print(f"Testing score: {test_score:.4f}")


#extract feature importances using permutation importance
from sklearn.inspection import permutation_importance
result = permutation_importance(rsf, 
                                X_test, 
                                y_test, n_repeats=16, random_state=42, n_jobs=8)

result_df = pd.DataFrame(
    {
        k: result[k]
        for k in (
            "importances_mean",
            "importances_std",
        )
    },
    index=X_test.columns,
)

result_df.sort_values(by="importances_mean", ascending=False).head(20)
plt.hist(result_df.importances_mean)
## add the importance of cluster to each gene
importance_df = pd.DataFrame([X.columns,labels]).T
importance_df = importance_df.rename(columns={0:'Gene Symbol', 1:'cluster_id'})
importance_df.head()
# add scores from importance
score = result_df.importances_mean.to_list()
cluster_id = range(n_clusters)
score_map = dict(zip(cluster_id,score))

importance_df['feature_importance'] = importance_df['cluster_id'].map(score_map)
importance_df.head()
top_genes_paper = [
    'S1PR1',
    'MIF',
    'H1R1B',
    'HDS17B10',
    'ABCB4',
    'MAOA',
    'OPRK1',
    'IP6K2',
    'TRPA1',
    'CTP11B1',
    'ABCC8',
    'DPAGT1',
    'KDR']
genes_of_interest = ['CDH3', 'ADORA2B', 'MMP14', 'IP6K2', 'HTR2A']
# Check the ranking of these genes in the importance_with_annotations dataframe
importance_df[importance_df['Gene Symbol'].isin(top_genes_paper)][['Gene Symbol', 'feature_importance']]

importance_df.sort_values(by='feature_importance', ascending=False).to_csv('feature_permute_scores.csv')
